# 00 · MRZ check digits — early exploration

**Document Autopilot · Blueprint week (early exploration code)**

Passports and many ID cards carry a *machine-readable zone* (MRZ): two or three lines of `OCR-B` text with built-in **check digits** (ICAO Doc 9303). This notebook tests the first, cheapest reader in the pipeline — the one that needs no AI at all:

1. Compute ICAO check digits and validate the two official ICAO specimen documents (passport TD3 and ID card TD1).
2. Show what check digits **can** and **cannot** catch when OCR misreads a character.
3. Show why an OCR "repair" is accepted only when exactly one fix passes — otherwise a person decides.

Pure Python, no downloads, runs anywhere (Colab, Kaggle, laptop). All data here is the fictional ICAO specimen person *Anna Maria Eriksson* of the fictional state *Utopia* (`UTO`) or randomly generated — **no real person's document is used anywhere in this project.**

In [1]:
import random
import string

# ICAO Doc 9303 Part 3: digits keep their value, A-Z = 10-35, filler '<' = 0; weights 7, 3, 1 repeating.
def char_value(c: str) -> int:
    if c.isdigit():
        return int(c)
    if c == "<":
        return 0
    if "A" <= c <= "Z":
        return ord(c) - ord("A") + 10
    raise ValueError(f"not an MRZ character: {c!r}")

def check_digit(field: str) -> str:
    weights = (7, 3, 1)
    total = sum(char_value(c) * weights[i % 3] for i, c in enumerate(field))
    return str(total % 10)

print("check_digit('L898902C3') =", check_digit("L898902C3"), " (ICAO specimen says 6)")

check_digit('L898902C3') = 6  (ICAO specimen says 6)


## 1 · Validate the official ICAO specimens

TD3 = passport data page (2 lines × 44 characters). TD1 = ID card (3 lines × 30 characters). Each field has its own check digit, plus one **composite** digit over several fields (only some formats define one — visas MRV-A/MRV-B do not).

In [2]:
def parse_td3(line1: str, line2: str) -> dict:
    assert len(line1) == 44 and len(line2) == 44, "TD3 lines are 44 characters"
    doc_no, doc_cd = line2[0:9], line2[9]
    dob, dob_cd = line2[13:19], line2[19]
    exp, exp_cd = line2[21:27], line2[27]
    opt, opt_cd = line2[28:42], line2[42]
    composite = line2[0:10] + line2[13:20] + line2[21:43]
    surname, _, given = line1[5:].partition("<<")
    return {
        "document_type": line1[0:2].replace("<", ""),
        "issuing_state": line1[2:5],
        "surname": surname.replace("<", " ").strip(),
        "given_names": given.replace("<", " ").strip(),
        "document_number": doc_no.rstrip("<"),
        "nationality": line2[10:13],
        "birth_date": dob,
        "sex": line2[20],
        "expiry_date": exp,
        "checks": {
            "document_number": check_digit(doc_no) == doc_cd,
            "birth_date": check_digit(dob) == dob_cd,
            "expiry_date": check_digit(exp) == exp_cd,
            "optional_data": check_digit(opt) == opt_cd,
            "composite": check_digit(composite) == line2[43],
        },
    }

def parse_td1(line1: str, line2: str, line3: str) -> dict:
    assert all(len(l) == 30 for l in (line1, line2, line3)), "TD1 lines are 30 characters"
    doc_no, doc_cd = line1[5:14], line1[14]
    dob, dob_cd = line2[0:6], line2[6]
    exp, exp_cd = line2[8:14], line2[14]
    composite = line1[5:30] + line2[0:7] + line2[8:15] + line2[18:29]
    surname, _, given = line3.partition("<<")
    return {
        "document_type": line1[0:2].replace("<", ""),
        "issuing_state": line1[2:5],
        "surname": surname.replace("<", " ").strip(),
        "given_names": given.replace("<", " ").strip(),
        "document_number": doc_no.rstrip("<"),
        "nationality": line2[15:18],
        "birth_date": dob,
        "sex": line2[7],
        "expiry_date": exp,
        "checks": {
            "document_number": check_digit(doc_no) == doc_cd,
            "birth_date": check_digit(dob) == dob_cd,
            "expiry_date": check_digit(exp) == exp_cd,
            "composite": check_digit(composite) == line2[29],
        },
    }

TD3 = ("P<UTOERIKSSON<<ANNA<MARIA<<<<<<<<<<<<<<<<<<<",
       "L898902C36UTO7408122F1204159ZE184226B<<<<<10")
TD1 = ("I<UTOD231458907<<<<<<<<<<<<<<<",
       "7408122F1204159UTO<<<<<<<<<<<6",
       "ERIKSSON<<ANNA<MARIA<<<<<<<<<<")

for name, result in (("TD3 passport", parse_td3(*TD3)), ("TD1 ID card", parse_td1(*TD1))):
    print(f"--- {name} ---")
    for key, value in result.items():
        print(f"  {key:16} {value}")
    print("  ALL CHECKS PASS:", all(result["checks"].values()))

--- TD3 passport ---
  document_type    P
  issuing_state    UTO
  surname          ERIKSSON
  given_names      ANNA MARIA
  document_number  L898902C3
  nationality      UTO
  birth_date       740812
  sex              F
  expiry_date      120415
  checks           {'document_number': True, 'birth_date': True, 'expiry_date': True, 'optional_data': True, 'composite': True}
  ALL CHECKS PASS: True
--- TD1 ID card ---
  document_type    I
  issuing_state    UTO
  surname          ERIKSSON
  given_names      ANNA MARIA
  document_number  D23145890
  nationality      UTO
  birth_date       740812
  sex              F
  expiry_date      120415
  checks           {'document_number': True, 'birth_date': True, 'expiry_date': True, 'composite': True}
  ALL CHECKS PASS: True


## 2 · What check digits catch — and what they miss

Simulate the most common OCR error: **one wrong character**. Corrupt one random character of a field (document number, birth date or expiry date) of 10,000 randomly generated fictional passports, and count how often the check digit notices.

In [3]:
random.seed(1378)
ALNUM = string.digits + string.ascii_uppercase

def random_doc_number() -> str:
    return "".join(random.choice(ALNUM) for _ in range(9))

def random_date() -> str:
    return f"{random.randint(0, 99):02d}{random.randint(1, 12):02d}{random.randint(1, 28):02d}"

def corrupt(field: str) -> str:
    i = random.randrange(len(field))
    pool = string.digits if field[i].isdigit() else ALNUM
    new = random.choice([c for c in pool if c != field[i]])
    return field[:i] + new + field[i + 1:]

stats = {"digit swapped": [0, 0], "letter swapped": [0, 0]}
for _ in range(10_000):
    field = random.choice([random_doc_number(), random_date(), random_date()])
    cd = check_digit(field)
    bad = corrupt(field)
    kind = "digit swapped" if any(a.isdigit() and a != b for a, b in zip(field, bad)) else "letter swapped"
    stats[kind][1] += 1
    stats[kind][0] += check_digit(bad) != cd

for kind, (caught, total) in stats.items():
    print(f"{kind:15}: caught {caught:5} of {total:5} = {caught / total:6.1%}")

digit swapped  : caught  7600 of  7600 = 100.0%
letter swapped : caught  2206 of  2400 =  91.9%


**Reading the result:** every wrong *digit* is caught (weights 7, 3 and 1 share no factor with 10). But a wrong *letter* can slip through — for example `A` (10) read as `K` (20) changes the weighted sum by a multiple of 10. A check-digit pass proves the line is *internally consistent*, not that it was *read correctly*. That is why the pipeline still re-reads every value with a second engine and compares the code lines with the printed text.

## 3 · OCR repair: accept only a unique fix

OCR often confuses look-alike characters (`0/O`, `1/I`, `8/B`, `5/S`, `2/Z`). If a field fails its check digit, try every look-alike substitution. **Accept a repair only if exactly one candidate passes** — and even then a person confirms it. Two or more passing candidates = ambiguous = a person decides.

In [4]:
from itertools import product

LOOKALIKES = {"0": "0O", "O": "O0", "1": "1I", "I": "I1", "8": "8B", "B": "B8",
              "5": "5S", "S": "S5", "2": "2Z", "Z": "Z2"}

def repair_candidates(read: str, expected_cd: str, digits_only: bool) -> list[str]:
    # A date field can only hold digits, so its look-alike letters have one possible meaning.
    options = []
    for c in read:
        alts = LOOKALIKES.get(c, c)
        options.append("".join(a for a in alts if a.isdigit()) if digits_only else alts)
    return sorted({"".join(p) for p in product(*options) if check_digit("".join(p)) == expected_cd})

def try_repair(read: str, expected_cd: str, digits_only: bool = False) -> str:
    if check_digit(read) == expected_cd:
        return f"{read}: passes as read"
    cands = repair_candidates(read, expected_cd, digits_only)
    if len(cands) == 1:
        return f"{read}: ONE repair passes -> propose {cands[0]} (a person still confirms)"
    if not cands:
        return f"{read}: no repair passes -> ask a person to type it"
    return f"{read}: {len(cands)} repairs pass {cands} -> AMBIGUOUS, ask a person"

print(try_repair("L898902C3", "6"))                     # document number read correctly
print(try_repair("74O8I2", "2", digits_only=True))      # birth date: O and I misread in a digits-only field
print(try_repair("L8989O2C3", "6"))                     # document number: letters allowed, so several fixes fit

L898902C3: passes as read
74O8I2: ONE repair passes -> propose 740812 (a person still confirms)
L8989O2C3: 3 repairs pass ['L898902C3', 'LB9890ZC3', 'LB9B902C3'] -> AMBIGUOUS, ask a person


## What this tells the build

- The MRZ lane is exact and almost free — it runs **first**, before any AI model.
- A check-digit pass is necessary, never sufficient: letters can be misread undetected, so every value is re-read by a second engine (Make It Yours phase).
- Repairs are proposals, never silent corrections. A digits-only date usually has one valid fix; a document number that mixes letters and digits often has several, so a person decides.

**Next (First Working Demo, week 11):** run pretrained OCR on 10 MIDV-2020 sample photos, find the MRZ lines, and feed them through `parse_td3` / `parse_td1` end to end.